In [1]:
%run ../setup.py
model = 'claude-sonnet-4-0'

In [2]:
import json

In [3]:
def add_message(messages, role, text):
  messages.append({
    'role': role,
    'content': text
  })

def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [4]:
def generate_dataset():
  prompt = """
Generate an evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects, each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
  {
    "task": "Description of task",
    "format': "json" or "python" or "regex",
    "solution_criteria": "Key criteria for evaluating the solution"
  },
  ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a single regex
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""
  messages = []
  add_message(messages, 'user', prompt)
  add_message(messages, 'assistant', "```json")
  text = chat(messages, stop_sequences=["```"])
  return json.loads(text)

In [5]:
dataset = generate_dataset()
print(dataset)

[{'task': "Create a JSON policy document that allows read-only access to a specific S3 bucket named 'my-data-bucket' for any IAM user or role", 'format': 'json', 'solution_criteria': 'Must include correct IAM policy structure with Version, Statement array, Effect set to Allow, Action for S3 read operations (s3:GetObject, s3:ListBucket), and Resource ARN pointing to the specified bucket and its objects'}, {'task': "Write a Python function that takes an AWS region name as input and returns the corresponding S3 endpoint URL (e.g., 'us-east-1' should return 'https://s3.us-east-1.amazonaws.com')", 'format': 'python', 'solution_criteria': 'Function must handle standard AWS regions, construct proper S3 endpoint URLs with https protocol, handle the special case of us-east-1 which can use both s3.amazonaws.com and s3.us-east-1.amazonaws.com, and return a string'}, {'task': "Create a regex pattern that validates AWS IAM role ARNs (Amazon Resource Names) in the format 'arn:aws:iam::123456789012:r

In [6]:
with open('dataset.json', 'w') as f:
    json.dump(dataset, f, indent=2)

In [7]:
def run_prompt(test_case):
  """Merges the prompt and test case input, then returns the result"""
  prompt = f"""
Please solve the following task:

{test_case['task']}

* Respond only with Python, JSON or a plain Regex
* Do not add any comments or commentary or explanation
"""

  messages = []
  add_message(messages, 'user', prompt)
  add_message(messages, 'assistant', "```code")
  output = chat(messages, stop_sequences=["```"])
  return output


In [8]:
def grade_by_model(test_case, output):
  eval_prompt = """
You are an expert code reviewer. Evaluate this AI-generated solution.

Task: {test_case['task']}
Solution: {output}
Criteria you should use to evaluate the solution: {test_case['solution_criteria']}

Provide your evaluation as a structured JSON object with:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement  
- "reasoning": A concise explanation of your assessment
- "score": A number between 1-10

Respond with JSON. Keep your response concise and direct.
Example response shape:
{{
  "strengths": string[],
  "weaknesses": string[],
  "reasoning": string,
  "score": number
}}
  """

  messages = []
  add_message(messages, 'user', eval_prompt)
  add_message(messages, 'assistant', "```json")
  eval_text = chat(messages, stop_sequences=["```"])
  return json.loads(eval_text)



In [9]:
import re
import ast

def validate_json(text):
  try:
    json.loads(text.strip())
    return 10
  except json.JSONDecodeError:
    return 0
  
def validate_python(text):
  try:
    ast.parse(text.strip())
    return 10
  except SyntaxError:
    return 0
  
def validate_regex(text):
  try:
    re.compile(text.strip())
    return 10
  except re.error:
    return 0
  
def grade_syntax(response, test_case):
  format = test_case['format']
  if format == 'json':
    return validate_json(response)
  elif format == 'python':
    return validate_python(response)
  else:
    return validate_regex(response)

In [10]:
def run_test_case(test_case):
  """Calls run_prompt, then grades the result"""
  output = run_prompt(test_case)

  model_grade = grade_by_model(test_case, output)
  model_score = model_grade["score"]
  reasoning = model_grade["reasoning"]

  syntax_grade = grade_syntax(output, test_case)
  
  score = (model_score + syntax_grade) / 2

  return {
    "output": output,
    "test_case": test_case,
    "score": score,
    "reasoning": reasoning
  }

In [11]:
from statistics import mean

def run_eval(dataset):
  """Loads the dataset and calls run_test_case with each test case"""
  results = []
 
  for test_case in dataset:
    result = run_test_case(test_case)
    results.append(result)

  average_score = mean(result["score"] for result in results)
  print(f"Average score: {average_score}")

  return results

In [12]:
with open('dataset.json', 'r') as f:
  dataset = json.load(f)

results = run_eval(dataset)

Average score: 6.666666666666667


In [13]:
print(json.dumps(results, indent=2))

[
  {
    "output": "\n{\n    \"Version\": \"2012-10-17\",\n    \"Statement\": [\n        {\n            \"Effect\": \"Allow\",\n            \"Principal\": \"*\",\n            \"Action\": [\n                \"s3:GetObject\",\n                \"s3:GetObjectVersion\",\n                \"s3:ListBucket\"\n            ],\n            \"Resource\": [\n                \"arn:aws:s3:::my-data-bucket\",\n                \"arn:aws:s3:::my-data-bucket/*\"\n            ]\n        }\n    ]\n}\n",
    "test_case": {
      "task": "Create a JSON policy document that allows read-only access to a specific S3 bucket named 'my-data-bucket' for any IAM user or role",
      "format": "json",
      "solution_criteria": "Must include correct IAM policy structure with Version, Statement array, Effect set to Allow, Action for S3 read operations (s3:GetObject, s3:ListBucket), and Resource ARN pointing to the specified bucket and its objects"
    },
    "score": 5.5,
    "reasoning": "Cannot evaluate the solution